# Semantic Segmentation with PyTorch and U-Net
> https://www.kaggle.com/code/shnakazawa/semantic-segmentation-with-pytorch-and-u-net#Define-Helper-Functions
> https://zenn.dev/aidemy/articles/a43ebe82dfbb8b

## Import Modules

In [1]:
import os
import numpy as np
import pandas as pd
import math
import time
import random
import gc
from pathlib import Path
import cv2
from tqdm.notebook import tqdm
from sklearn.model_selection import KFold

# Image augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Modeling
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# Visualization
import matplotlib.pyplot as plt
%matplotlib inline

print(f'PyTorch version {torch.__version__}')
print(f'Albumentations version {A.__version__}')

/Users/atsushi-takahiro@cookpad.com/hobby/kaggle_notebook/.venv/lib/python3.12/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


PyTorch version 2.7.0
Albumentations version 2.0.7


## Set Configs
Separately configuring settings such as objectives for running, file paths, hyperparameters, and other parameters can greatly aid in maintaining a clear and organized workflow.

In [4]:
RUN_EDA = True
RUN_TRAINING = True
TRAIN_ALL = False # If true, train with all data and output a single model. If False, run cross-validation and output multiple models.
FOLD_NUM = 5 # For cross-validation
EPOCHS = 20 # Training cycle
RUN_INFERENCE = False

# Directory setting
DATA_DIR = '/Users/atsushi-takahiro@cookpad.com/hobby/datasets/sartorius-cell-instance-segmentation/'
MODEL_DIR = '/Users/atsushi-takahiro@cookpad.com/hobby/datasets/working/'
IMG_SAVE_DIR = '/Users/atsushi-takahiro@cookpad.com/hobby/datasets/working/'

# PyTorch variables
SEED = 42
NUM_WORKERS = 2
BATCH_SIZE = 8
WEIGHT_DECAY = 0.0001
LR = 0.0001
MOMENTUM = 0.9

# Threshold for mask prediction
THRESHOLD = 0.3

# Set device
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu') 
print(f'Using {device} device')

Using mps device


## Define Helper Functions
Defining reusable functions at the beginning of a Jupyter Notebook can result in code that is cleaner, more organized, and more efficient.

In [3]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

# Set seed
seed_everything(SEED)


def show_gpu_memory(device):
    print(f"Allocated GPU memory: {torch.cuda.memory_allocated(device) / 1024 / 1024:.2f} MB")
    print(f"Cached GPU memory: {torch.cuda.memory_cached(device) / 1024 / 1024:.2f} MB")    

    
def load_img(path):
    img_bgr = cv2.imread(path)
    img_rgb = img_bgr[:, :, ::-1]
    return img_rgb


def group_bboxes(df):
    df_ = df.copy()
    df_['segment_count'] = 1
    df_ = df_.groupby(['id', 'width', 'height', 'cell_type']).count().reset_index()
    return_df = df_[['id', 'width', 'height', 'cell_type', 'segment_count']]
    return return_df


def create_gallery(array, ncols=3):
    """Display multiple images in a gallery style.
    Source: https://www.amazon.co.jp/Data-Analysis-Machine-Learning-Kaggle-ebook/dp/B09F3STL34/
    
    Args:
        array (numpy.ndarray): array of images.
        ncols (int, optional): Num of columns. Defaults to 3.

    Returns:
        numpy.ndarray: One concatenated image.
    """    
    nindex, height, width, intensity = array.shape
    nrows = nindex//ncols
    assert nindex == nrows * ncols
    result = (array.reshape(nrows, ncols, height, width, intensity)
        .swapaxes(1,2)
        .reshape(height*nrows, width*ncols, intensity))
    return result


def decode_rle(rle, height, width):
    """RLE to image
    modified from: https://www.kaggle.com/paulorzp/run-length-encode-and-decode

    Args:
        rle (str): mask with run length encoding.
        height (int): return image height.
        width (int): return image width.
        brightness (int): brightness of the pixel. Default to 1.

    Returns:
        np.ndarray: 1(b) - mask, 0 - background.
    """    
    s = rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(height * width, dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1 # brightness
    return img.reshape((height, width)) 


def create_mask_image(image, masks):
    """Create a mask image from RLE.

    Args:
        image (numpy.ndarray): array of images.
        masks (list): List with RLE-encoded mask information.
        b (int): brightness of the pixel. Default to 1.
    
    Returns:
        numpy.ndarray: 1(b) - mask, 0 - background.
    """    
    
    s = image.shape
    h = s[0]
    w = s[1]
    mask_image = np.zeros((h,w))
    for mask in masks:
        mask_image += decode_rle(mask, h, w)
    mask_image = mask_image.clip(0, 1)
    return mask_image


def show_validation_score(train_loss_list, valid_loss_list, save=False, save_dir=IMG_SAVE_DIR, save_name='segmentation_validation_score.png'):
    fig = plt.figure(figsize=(10,10))
    for i in range(FOLD_NUM):
        train_loss = train_loss_list[i]
        valid_loss = valid_loss_list[i]
        
        ax = fig.add_subplot(math.ceil(np.sqrt(FOLD_NUM)), math.ceil(np.sqrt(FOLD_NUM)), i+1, title=f'Fold {i+1}')
        ax.plot(range(EPOCHS), train_loss, c='orange', label='train')
        ax.plot(range(EPOCHS), valid_loss, c='blue', label='valid')
        ax.set_xlabel('epoch')
        ax.set_ylabel('loss')
        ax.legend()
    
    plt.tight_layout()
    if save:
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(save_dir+save_name)
    else:
        plt.show()


def encode_rle(predicted_img):
    predicted_img = (predicted_img > THRESHOLD).astype(int)
    height, width = predicted_img.shape
    
    # Get the index of the masked pixel
    pixels = predicted_img.copy()
    pixels_list = []
    for y in range(height):
        for x in range(width):
            if pixels[y][x] != 0:
                pixels_list.append(y * width + x)

    # RLE encoding
    rle_list = []
    start = pixels_list[0]
    count = 1
    for i in range(1, len(pixels_list)):
        if pixels_list[i] == pixels_list[i-1] + 1:
            count += 1
        else:
            rle_list.extend([start, count])
            start = pixels_list[i]
            count = 1
    rle_list.extend([start, count])
    
    rle_str = [str(x) for x in rle_list]
    return ' '.join(rle_str)

## Load and Reshape a Table

In [5]:
df = pd.read_csv(DATA_DIR + 'train.csv')
df.head()

,id,annotation,width,height,cell_type,plate_time,sample_date,sample_id,elapsed_timedelta
0,0030fd0e6378,118145 6 118849 7 119553 8 120257 8 120961 9 1...,704,520,shsy5y,11h30m00s,2019-06-16,shsy5y[diff]_E10-4_Vessel-714_Ph_3,0 days 11:30:00
1,0030fd0e6378,189036 1 189739 3 190441 6 191144 7 191848 8 1...,704,520,shsy5y,11h30m00s,2019-06-16,shsy5y[diff]_E10-4_Vessel-714_Ph_3,0 days 11:30:00
2,0030fd0e6378,173567 3 174270 5 174974 5 175678 6 176382 7 1...,704,520,shsy5y,11h30m00s,2019-06-16,shsy5y[diff]_E10-4_Vessel-714_Ph_3,0 days 11:30:00
3,0030fd0e6378,196723 4 197427 6 198130 7 198834 8 199538 8 2...,704,520,shsy5y,11h30m00s,2019-06-16,shsy5y[diff]_E10-4_Vessel-714_Ph_3,0 days 11:30:00
4,0030fd0e6378,167818 3 168522 5 169225 7 169928 8 170632 9 1...,704,520,shsy5y,11h30m00s,2019-06-16,shsy5y[diff]_E10-4_Vessel-714_Ph_3,0 days 11:30:00


In [6]:
grouped_df = group_bboxes(df)
grouped_df.head()

,id,width,height,cell_type,segment_count
0,0030fd0e6378,704,520,shsy5y,395
1,0140b3c8f445,704,520,astro,108
2,01ae5a43a2ab,704,520,cort,36
3,026b3c2c4b32,704,520,cort,42
4,029e5b3b89c7,704,520,cort,34
